In [59]:
import pandas as pd
import numpy as np
import arcticdb as adb
import nasdaqdatalink as ndl
import matplotlib.pyplot as plt
from jedi.inference.utils import to_list
%matplotlib inline
import time
from tqdm import tqdm
from collections import defaultdict

In [55]:
START = "2000-01-01"
END = "2026-02-15"

ARCTIC_URI = "lmdb://./database/arctic"

In [56]:

ac = adb.Arctic(ARCTIC_URI)

20260515 20:38:18.830698 50328 W arcticdb | LMDB path at /home/sysadmin/PycharmProjects/PM/database/arctic/ has already been opened in this process which is not supported by LMDB. You should only open a single Arctic instance over a given LMDB path. To continue safely, you should delete this Arctic instance and any others over the LMDB path in this process and then try again. Current process ID=[50328]


In [57]:
etf_lib = ac["etfs_lib"]
stocks_lib = ac["stocks_lib"]
removed_lib = ac["removed_stocks"]

In [58]:
print(sorted(stocks_lib.list_symbols()))
print(sorted(removed_lib.list_symbols()))

['A', 'AAPL', 'ABBV', 'ABNB', 'ABT', 'ACGL', 'ACN', 'ADBE', 'ADI', 'ADM', 'ADP', 'ADSK', 'AEE', 'AEP', 'AES', 'AFL', 'AIG', 'AIZ', 'AJG', 'AKAM', 'ALB', 'ALGN', 'ALL', 'ALLE', 'AMAT', 'AMCR', 'AMD', 'AME', 'AMGN', 'AMP', 'AMT', 'AMZN', 'ANET', 'AON', 'AOS', 'APA', 'APD', 'APH', 'APO', 'APP', 'APTV', 'ARE', 'ARES', 'ATO', 'AVB', 'AVGO', 'AVY', 'AWK', 'AXON', 'AXP', 'AZO', 'BA', 'BAC', 'BALL', 'BAX', 'BBY', 'BDX', 'BEN', 'BF.B', 'BG', 'BIIB', 'BK', 'BKNG', 'BKR', 'BLDR', 'BLK', 'BMY', 'BR', 'BRK.B', 'BRO', 'BSX', 'BX', 'BXP', 'C', 'CAG', 'CAH', 'CARR', 'CASY', 'CAT', 'CB', 'CBOE', 'CBRE', 'CCI', 'CCL', 'CDNS', 'CDW', 'CEG', 'CF', 'CFG', 'CHD', 'CHRW', 'CHTR', 'CI', 'CIEN', 'CINF', 'CL', 'CLX', 'CMCSA', 'CME', 'CMG', 'CMI', 'CMS', 'CNC', 'CNP', 'COF', 'COHR', 'COIN', 'COO', 'COP', 'COR', 'COST', 'CPAY', 'CPB', 'CPRT', 'CPT', 'CRH', 'CRL', 'CRM', 'CRWD', 'CSCO', 'CSGP', 'CSX', 'CTAS', 'CTSH', 'CTVA', 'CVNA', 'CVS', 'CVX', 'D', 'DAL', 'DASH', 'DD', 'DDOG', 'DE', 'DECK', 'DELL', 'DG', 'DGX',

In [49]:
aapl = stocks_lib.read("AAPL").metadata


In [50]:
aapl


{'permaticker': 199059,
 'sector': 'Technology',
 'firstpricedate': Timestamp('1986-01-01 00:00:00'),
 'lastpricedate': Timestamp('2026-05-15 00:00:00')}

In [53]:
aal = removed_lib.read("AAL").metadata
aal


{'permaticker': 199960,
 'sector': 'Industrials',
 'firstpricedate': Timestamp('2013-12-09 00:00:00'),
 'lastpricedate': Timestamp('2026-05-15 00:00:00')}

In [62]:
def group_lib(lib, key):
    buckets = defaultdict(list)
    for sym in lib.list_symbols():
        meta = lib.read_metadata(sym).metadata
        buckets[meta.get(key)].append(sym)
    return dict(buckets)



In [64]:
by_sector = group_lib(stocks_lib, "sector")
by_sector

{'Technology': ['JKHY',
  'SWKS',
  'ON',
  'ANET',
  'TTD',
  'INTC',
  'PTC',
  'GRMN',
  'MSFT',
  'MU',
  'WDC',
  'ROP',
  'CTSH',
  'IT',
  'FFIV',
  'DDOG',
  'CRWD',
  'AKAM',
  'LITE',
  'CPAY',
  'PLTR',
  'UBER',
  'ACN',
  'DELL',
  'AMAT',
  'FTNT',
  'CDW',
  'AMD',
  'MSI',
  'NOW',
  'NTAP',
  'SNDK',
  'WDAY',
  'LDOS',
  'FICO',
  'TRMB',
  'XYZ',
  'HPE',
  'GDDY',
  'VRSN',
  'TDY',
  'STX',
  'JBL',
  'SATS',
  'CSCO',
  'SNPS',
  'BR',
  'ADI',
  'AAPL',
  'TXN',
  'LRCX',
  'FISV',
  'PANW',
  'MCHP',
  'TYL',
  'INTU',
  'TER',
  'TEL',
  'APP',
  'ADSK',
  'ZBRA',
  'KLAC',
  'NXPI',
  'CIEN',
  'EPAM',
  'GEN',
  'APH',
  'FSLR',
  'ADBE',
  'QCOM',
  'ORCL',
  'Q',
  'NVDA',
  'CDNS',
  'HPQ',
  'AVGO',
  'FIS',
  'MPWR',
  'IBM',
  'COHR',
  'GLW',
  'FTV',
  'SMCI',
  'CRM',
  'KEYS'],
 'Financial Services': ['IBKR',
  'ARES',
  'MET',
  'FITB',
  'MTB',
  'BEN',
  'HBAN',
  'WFC',
  'MS',
  'ALL',
  'WRB',
  'CINF',
  'AFL',
  'SYF',
  'BLK',
  'WTW',
  'C

In [65]:
def check_price_records(lib):
    records = []
    for item in lib.read_batch(lib.list_symbols()):
        df = item.data
        if df.empty:
            records.append((item.symbol, None, None, 0))
            continue
        records.append((item.symbol, df.index.min(), df.index.max(), len(df)))
    coverage = pd.DataFrame(records, columns=["ticker", "start", "end", "count"])
    coverage["start"] = pd.to_datetime(coverage["start"])
    coverage["end"] = pd.to_datetime(coverage["end"])
    return coverage



In [69]:
cov = check_price_records(stocks_lib)
cov["start"].dt.year.value_counts().sort_index()    # IPO years
print(len(cov[cov["count"] == 0]))
print(len(cov[cov["start"] > "2000-01-01"]))


0
503


In [70]:
cov[cov["start"] > "2000-01-01"]

,ticker,start,end,count
0,JKHY,2000-01-03,2026-02-13,6569
1,IBKR,2007-05-04,2026-02-13,4726
2,LOW,2000-01-03,2026-02-13,6569
3,EVRG,2018-04-25,2026-02-13,1963
4,SWKS,2000-01-03,2026-02-13,6569
...,...,...,...,...
498,FTV,2016-07-05,2026-02-13,2418
499,PNC,2000-01-03,2026-02-13,6569
500,SMCI,2007-03-29,2026-02-13,4751
501,CRM,2004-06-23,2026-02-13,5447
